In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from helpers import *
from dotenv import load_dotenv

load_dotenv()

# ---- Surprise SVD ----
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split as surprise_split
from surprise import accuracy

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df_ratings[['userId', 'movieId', 'rating']], reader)

trainset, testset = surprise_split(data, test_size=0.2, random_state=42)

model = SVD()
model.fit(trainset)
predictions = model.test(testset)

print("=== Surprise SVD ===")
surprise_rmse = accuracy.rmse(predictions)
surprise_mae = accuracy.mae(predictions)

=== Surprise SVD ===
RMSE: 0.8828
MAE:  0.6784


In [2]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=5, threshold=4.0):
    # Group predictions by user
    user_est_true = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))
    
    precisions = dict()
    recalls = dict()
    
    for uid, user_ratings in user_est_true.items():
        # Sort by estimated rating (highest first)
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        
        # Number of relevant items (user truly liked)
        n_rel = sum((true_r >= threshold) 
                    for (_, true_r) in user_ratings)
        
        # Number of recommended items in top k
        n_rec_k = sum((est >= threshold) 
                      for (est, _) in user_ratings[:k])
        
        # Number of relevant AND recommended in top k
        n_rel_and_rec_k = sum(
            ((true_r >= threshold) and (est >= threshold))
            for (est, true_r) in user_ratings[:k]
        )
        
        # Precision@K
        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        
        # Recall@K
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0
    
    # Average across all users
    avg_precision = sum(precisions.values()) / len(precisions)
    avg_recall = sum(recalls.values()) / len(recalls)
    
    return avg_precision, avg_recall


# Run evaluation
precision, recall = precision_recall_at_k(predictions, k=5, threshold=4.0)

print(f"Precision@5: {precision:.4f}")
print(f"Recall@5: {recall:.4f}")

Precision@5: 0.5776
Recall@5: 0.2278


In [3]:
def compute_metrics():
    """Compute evaluation metrics from Supabase feedback."""
    try:
        res = supabase.table("feedback").select("*").execute()
        rows = res.data
    except Exception as e:
        print(f"Metrics read failed: {e}")
        return None

    if not rows:
        return None

    df = pd.DataFrame(rows)

    shown   = df[df["action"] == "shown"]
    accepts = df[df["action"] == "accept"]
    rates   = df[df["action"] == "rate"]

    total_shown = pd.to_numeric(shown["reason"], errors="coerce").sum()
    acceptance_rate = (len(accepts) / total_shown * 100) if total_shown else 0

    decision_times = pd.to_numeric(accepts["reason"], errors="coerce").dropna()
    avg_decision = decision_times.mean() if len(decision_times) else 0

    satisfaction = pd.to_numeric(rates["reason"], errors="coerce").dropna()
    avg_satisfaction = satisfaction.mean() if len(satisfaction) else 0

    ces = (avg_satisfaction / avg_decision) if avg_decision else 0

    return {
        "acceptance_rate": round(acceptance_rate, 1),
        "avg_decision_time": round(avg_decision, 1),
        "avg_satisfaction": round(avg_satisfaction, 2),
        "ces": round(ces, 3),
        "total_shown": int(total_shown),
        "total_accepts": len(accepts),
    }

In [4]:
print(compute_metrics())

{'acceptance_rate': 41.7, 'avg_decision_time': 81.6, 'avg_satisfaction': 3.0, 'ces': 0.037, 'total_shown': 12, 'total_accepts': 5}
